In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_curve, auc, roc_curve
from sklearn.utils import shuffle
from datetime import timedelta
from src.common.feature_dtypes import expected_dtypes
from tqdm import tqdm
import random
import os
import boto3
import tempfile
import io

In [2]:

# Define S3 info
bucket = 'kehmisjan2025'

# Initialize boto3 client
s3 = boto3.client('s3')
buffer = io.BytesIO()
s3.download_fileobj(bucket, 'rtc0829.parquet', buffer)
buffer.seek(0)  # Move to the start of the buffer
df = pd.read_parquet(buffer)


In [3]:
# create variables is_linked if ReasonforMissedAppt is not null
df['is_linked'] = df['ReasonforMissedAppt'].notnull()

# get count by is_linked
df['is_linked'].value_counts()

is_linked
False    663624
True      22126
Name: count, dtype: int64

In [4]:
# filter to emr in kenyamer and ecare
df = df[df["emr"].isin(["kenyaemr"])]
df = df.drop(columns=["emr"])

df.columns = df.columns.str.lower().str.replace(" ", "_")

# ensure columns are right dtypes
for col, dtype in expected_dtypes.items():
    if col in df.columns:
        if dtype in [float, "float", "float64", int, "int", "int64"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        else:
            df[col] = df[col].astype(dtype)

In [5]:
cols_to_keep = ['cascadestatus', 'visittype', 'visitby', 'tcareason', 'pregnant',
       'breastfeeding', 
       'stabilityassessment', 'differentiatedcare', 'whostage',
       'adherence', 'sex', 'age', 'maritalstatus', 'educationlevel',
       'occupation', 'bmi', 'regimen_switch',  'is_friday', 'daystonextappointment', 'timeonart',
       'firstvisit', 'lastvd', 'late', 'late14', 'late30',
       'lateness_last3', 'lateness_last5', 'lateness_last10', 'late_last3',
       'late_last5', 'late_last10', 'late14_last3', 'late14_last5',
       'late14_last10', 'late30_last3', 'late30_last5', 'late30_last10',
       'optimizedhivregimen', 'most_recent_vl', 'ahd', 'kephlevel',
       'facilitytypecategory', 'ownertype', 'men_knowledge',
       'women_knowledge', 'men_heardaids', 'men_highrisksex',
       'men_highrisksex_multi', 'men_sexnotwithpartner', 'men_sexpartners',
       'men_nevertested', 'men_testedrecent', 'men_sti', 'women_heardaids',
       'women_highrisksex', 'women_highrisksex_multi',
       'women_sexnotwithpartner', 'women_sexpartners', 'women_nevertested',
       'women_testedrecent', 'women_sti', 'rolling_weighted_noshow',
       'rolling_weighted_dayslate', 'is_linked'

]

In [6]:
# create df_ipw as df subset by cols to keep
print(df.shape)
df_ipw = df[cols_to_keep]
print(df_ipw.shape)

(614102, 87)
(614102, 64)


In [7]:
import pandas as pd
from sklearn.impute import SimpleImputer
import numpy as np

exclude_cols = ['is_linked']

# Numeric columns
num_vars = df_ipw.select_dtypes(include=['int32', 'int64', 'float64', 'bool']).columns.tolist()
num_vars = [c for c in num_vars if c not in exclude_cols]

# Convert bool to int
for c in num_vars:
    if df_ipw[c].dtype == 'bool':
        df_ipw[c] = df_ipw[c].astype(int)

# Impute numeric variables safely
num_imputer = SimpleImputer(strategy='median')
num_imputed_array = num_imputer.fit_transform(df_ipw[num_vars])

# Make sure number of columns matches
num_imputed_df = pd.DataFrame(num_imputed_array, columns=num_vars, index=df_ipw.index)
df_ipw[num_vars] = num_imputed_df

# Categorical columns
cat_vars = df_ipw.select_dtypes(include=['object', 'category']).columns.tolist()
cat_vars = [c for c in cat_vars if c not in exclude_cols]

if cat_vars:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    cat_imputed_array = cat_imputer.fit_transform(df_ipw[cat_vars])
    cat_imputed_df = pd.DataFrame(cat_imputed_array, columns=cat_vars, index=df_ipw.index)
    df_ipw[cat_vars] = cat_imputed_df

    # One-hot encode
    df_ipw = pd.get_dummies(df_ipw, columns=cat_vars, drop_first=True)

print(df_ipw.shape)


/tmp/ipykernel_19967/3571196172.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ipw[num_vars] = num_imputed_df
/tmp/ipykernel_19967/3571196172.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ipw[cat_vars] = cat_imputed_df


(614102, 101)


In [8]:
# x_prop drops is_linked
X_prop = df_ipw.drop(columns=['is_linked'])
y_prop = df_ipw['is_linked'].astype(int)
print(X_prop.shape, y_prop.shape)

(614102, 100) (614102,)


In [9]:
# Step 1: Fit propensity model
from sklearn.linear_model import LogisticRegression
prop_model = LogisticRegression(max_iter=5000)
prop_model.fit(X_prop, y_prop)

# Step 2: Predict propensity scores for *all* patients
df_ipw['propensity'] = prop_model.predict_proba(X_prop)[:,1]

# Step 3: Calculate stabilized weights
p_linked = df_ipw['is_linked'].mean()   # overall fraction linked
df_ipw['stabilized_weight'] = np.where(
    df_ipw['is_linked'] == 1,
    p_linked / df_ipw['propensity'],
    (1 - p_linked) / (1 - df_ipw['propensity'])
)

# Step 4: Cap extreme weights
df_ipw['stabilized_weight'] = df_ipw['stabilized_weight'].clip(upper=10)

# hstack two columns from df - ReasonforMissedAppt and TracingType
df_ipw['reasonformissedappt'] = df['reasonformissedappt'].astype(str) 
df_ipw['tracingtype'] = df['tracingtype'].astype(str) 


# Step 5: Keep only linked patients for downstream modeling
linked = df_ipw[df_ipw['is_linked'] == 1].copy()

In [10]:
# X should contain all features except for reasonformissedappt, tracingtype, stabilized_weight and propensity
X = linked.drop(['reasonformissedappt', 'tracingtype', 'stabilized_weight', 'propensity'], axis=1)       # numeric + one-hot encoded categorical features
y_reason = linked['reasonformissedappt']  # multi-class
y_tracing = linked['tracingtype']  # multi-class
sample_weights = linked['stabilized_weight']

In [11]:
# convert y_reason to Other where y_reason is "Client has covid-19 infection"
y_reason = y_reason.replace("Client has covid-19 infection", "Other")

y_reason.value_counts()

reasonformissedappt
Client travelled                                6068
Other                                           4809
Client could not get an off from work/school    3559
Client refilled drugs from another facility     2400
Client has enough drugs                         2204
Client forgot clinic dates                      1192
Client sick at home/admitted                     931
Client stopped medications                       810
Client is sharing drugs with partner             153
Name: count, dtype: int64

In [16]:
y_reason.value_counts()

reasonformissedappt
Client travelled                                6068
Other                                           4809
Client could not get an off from work/school    3559
Client refilled drugs from another facility     2400
Client has enough drugs                         2204
Client forgot clinic dates                      1192
Client sick at home/admitted                     931
Client stopped medications                       810
Client is sharing drugs with partner             153
Name: count, dtype: int64

In [13]:
from sklearn.model_selection import train_test_split

X_train_r, X_test_r, y_train_r, y_test_r, w_train_r, w_test_r = train_test_split(
    X, y_reason, sample_weights, test_size=0.2, random_state=42, stratify=y_reason
)

X_train_t, X_test_t, y_train_t, y_test_t, w_train_t, w_test_t = train_test_split(
    X, y_tracing, sample_weights, test_size=0.2, random_state=42, stratify=y_tracing
)


In [14]:
from sklearn.ensemble import RandomForestClassifier

# Reason for missed appointment
clf_reason = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
clf_reason.fit(X_train_r, y_train_r, sample_weight=w_train_r)

# Tracing type
clf_tracing = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
clf_tracing.fit(X_train_t, y_train_t, sample_weight=w_train_t)


RandomForestClassifier(class_weight='balanced', max_depth=12, n_estimators=500,
                       n_jobs=-1, random_state=42)

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import make_scorer, precision_recall_fscore_support, roc_auc_score, classification_report
import numpy as np

# ----- 1. Define your XGBoost model -----
xgb_clf = xgb.XGBClassifier(
    objective="multi:softprob",  # or "multi:softprob" for multi-class
    eval_metric="logloss",        # XGBoost’s internal metric
    use_label_encoder=False
)

# ----- 2. Set up hyperparameter grid -----
param_grid = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
    "n_estimators": [100, 300, 500],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

# ----- 3. Cross-validation setup -----
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ----- 4. Define scoring -----
def pr_auc_scorer(estimator, X, y_true, sample_weight=None):
    y_pred_proba = estimator.predict_proba(X)[:, 1]
    return roc_auc_score(y_true, y_pred_proba, sample_weight=sample_weight)

scoring = {
    "roc_auc": make_scorer(roc_auc_score, needs_proba=True),
    "precision": make_scorer(lambda y, y_pred: precision_recall_fscore_support(
        y, y_pred, average="weighted")[0]),
    "recall": make_scorer(lambda y, y_pred: precision_recall_fscore_support(
        y, y_pred, average="weighted")[1])
}

# ----- 5. Run Grid Search with sample weights -----
grid_search = GridSearchCV(
    estimator=xgb_clf,
    param_grid=param_grid,
    scoring=scoring,
    refit="roc_auc",    # optimize for ROC-AUC
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_r, y_train_r, sample_weight=w_train_r)

# ----- 6. Best model -----
best_model = grid_search.best_estimator_
print("Best params:", grid_search.best_params_)

# ----- 7. Evaluate on test set -----
y_pred = best_model.predict(X_test_r)
y_proba = best_model.predict_proba(X_test_r)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test_r, y_pred, sample_weight=w_test_r, digits=3))

print("ROC-AUC:", roc_auc_score(y_test_r, y_proba, sample_weight=w_test_r))


Reason for missed appointment:
                                              precision    recall  f1-score   support

Client could not get an off from work/school       0.41      0.37      0.39       712
                  Client forgot clinic dates       0.15      0.13      0.14       238
                     Client has enough drugs       0.23      0.36      0.28       441
        Client is sharing drugs with partner       0.00      0.00      0.00        31
 Client refilled drugs from another facility       0.22      0.22      0.22       480
                Client sick at home/admitted       0.10      0.15      0.12       186
                  Client stopped medications       0.13      0.28      0.18       162
                            Client travelled       0.36      0.29      0.32      1214
                                       Other       0.42      0.35      0.38       962

                                    accuracy                           0.30      4426
                     